# LeetCode 3762 — Minimum Operations to Equalize Subarrays bằng Python

- Link: https://leetcode.com/problems/minimum-operations-to-equalize-subarrays/
- Độ khó: **Hard**
- Chủ đề: Số dư modulo, **trung vị** tối thiểu tổng khoảng cách, truy vấn đoạn tĩnh, **Persistent Segment Tree** (cây phân đoạn bền vững), thuật toán Mo

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def minOperations(self, nums: List[int], k: int, queries: List[List[int]]) -> List[int]:
```

## 1. Mục tiêu học

1. Nhận ra điều kiện khả thi: mọi phần tử trong đoạn cùng **số dư khi chia cho `k`** — kiểm tra `O(1)` bằng tổng tiền tố.
2. Quy về thương `q_i = nums[i] // k`: số thao tác để đưa `a` về `b` là `|q_a − q_b|`.
3. Dùng tính chất **trung vị** làm cực tiểu `Σ|q_i − t|`.
4. Trả lời "phần tử nhỏ thứ `c`" và "tổng `c` phần tử nhỏ nhất" của một đoạn bằng **persistent segment tree** trong `O(log n)`.
5. So sánh với vét cạn (sắp xếp mỗi truy vấn) và thuật toán **Mo** offline.

## 2. Đề bài & ví dụ

Cho `nums`, `k`. Mỗi thao tác tăng hoặc giảm **một** phần tử đúng `k`. Với mỗi truy vấn `[l, r]`, tìm số thao tác tối thiểu để mọi phần tử của `nums[l..r]` bằng nhau, hoặc `-1` nếu không thể.

**Ví dụ 1:** `nums = [1,4,7], k = 3, queries = [[0,1],[0,2]]` → `[1, 2]`.

**Ví dụ 2:** `nums = [1,2,4], k = 2, queries = [[0,2],[0,0],[1,2]]` → `[-1, 0, 1]`.

**Ràng buộc:** `n, q ≤ 4·10^4`, `1 ≤ nums[i], k ≤ 10^9`.

In [1]:
from typing import List
import random, time, math

examples = [
    ([1, 4, 7], 3, [[0, 1], [0, 2]], [1, 2]),
    ([1, 2, 4], 2, [[0, 2], [0, 0], [1, 2]], [-1, 0, 1]),
]
for a, k, qs, exp in examples:
    print(a, k, qs, "-> kỳ vọng", exp)

[1, 4, 7] 3 [[0, 1], [0, 2]] -> kỳ vọng [1, 2]
[1, 2, 4] 2 [[0, 2], [0, 0], [1, 2]] -> kỳ vọng [-1, 0, 1]


## 3. Kiến thức nền

### 3.1. Số dư bất biến

Cộng/trừ `k` không đổi `x mod k`. Hai phần tử chỉ có thể bằng nhau nếu cùng số dư; khi cùng số dư `ρ`, `x = q·k + ρ` và mọi đích chung phải có dạng `t·k + ρ`, chi phí đưa `x` về đó là `|q − t|`.

In [2]:
k = 3
for x in [1, 4, 7, 8]:
    print(x, "mod", k, "=", x % k, "| thương =", x // k)

1 mod 3 = 1 | thương = 0
4 mod 3 = 1 | thương = 1
7 mod 3 = 1 | thương = 2
8 mod 3 = 2 | thương = 2


### 3.2. Trung vị cực tiểu tổng khoảng cách

`f(t) = Σ|q_i − t|` là hàm lồi, tuyến tính từng khúc; độ dốc tại `t` = `#{q_i < t} − #{q_i > t}`. Độ dốc đổi dấu tại **trung vị** ⇒ `t = trung vị` là tối ưu (với số phần tử chẵn, mọi `t` giữa hai trung vị đều tối ưu).

In [3]:
q = [0, 1, 2, 9]
for t in range(-1, 11):
    print(t, sum(abs(x - t) for x in q), end=" | ")
print("\ntrung vị dưới:", sorted(q)[(len(q) - 1) // 2])

-1 16 | 0 12 | 1 10 | 2 10 | 3 12 | 4 14 | 5 16 | 6 18 | 7 20 | 8 22 | 9 24 | 10 28 | 
trung vị dưới: 1


### 3.3. Công thức chi phí bằng tổng `c` phần tử nhỏ nhất

Gọi `L = r − l + 1`, `c = (L + 1) // 2`, `m` = phần tử nhỏ thứ `c` (trung vị), `S_low` = tổng `c` phần tử nhỏ nhất, `S` = tổng tất cả. Khi đó:

`cost = (m·c − S_low) + ((S − S_low) − m·(L − c))`.

Vậy chỉ cần trả lời nhanh hai câu hỏi: **phần tử nhỏ thứ `c`** và **tổng `c` phần tử nhỏ nhất** trong đoạn.

### 3.4. Persistent Segment Tree

Dựng cây đếm (count + sum) theo **giá trị** cho tiền tố `nums[0..i−1]`, mỗi phiên bản `root[i]` chỉ tạo thêm `O(log n)` nút mới (chia sẻ phần còn lại). Đoạn `[l, r]` ứng với hiệu hai phiên bản `root[r+1] − root[l]` — đi xuống đồng thời hai cây để tìm phần tử thứ `c`.

## 4. Ý tưởng nền tảng / cách nghĩ trực giác

1. **Khả thi?** Đặt `diff[i] = 1` nếu `nums[i] % k ≠ nums[i−1] % k`. Đoạn khả thi ⇔ `Σ diff[l+1..r] = 0` (tổng tiền tố).
2. **Chi phí:** với `q_i = nums[i] // k`, đáp án = `Σ|q_i − median|` (mục 3.2, 3.3).

Ví dụ 1 truy vấn `[0,2]`: `q = [0,1,2]`, trung vị 1 ⇒ `1 + 0 + 1 = 2`. ✔ Ví dụ 2 truy vấn `[0,2]`: số dư `1, 0, 0` ⇒ `-1`. ✔

## 5. Cách 1 — Brute Force: sắp xếp từng đoạn

In [4]:
def brute(nums, k, queries):
    ans = []
    for l, r in queries:
        seg = nums[l:r + 1]
        if len({x % k for x in seg}) > 1:
            ans.append(-1); continue
        q = sorted(x // k for x in seg)
        m = q[(len(q) - 1) // 2]
        ans.append(sum(abs(x - m) for x in q))
    return ans

for a, k, qs, exp in examples:
    print(brute(a, k, qs), "kỳ vọng", exp)

[1, 2] kỳ vọng [1, 2]
[-1, 0, 1] kỳ vọng [-1, 0, 1]


Để chắc chắn "trung vị là tối ưu", thêm một kiểm chứng **độc lập** thử mọi đích (mọi giá trị trong đoạn — đủ vì hàm lồi tuyến tính từng khúc đạt cực tiểu tại một điểm gãy):

In [5]:
def brute_all_targets(nums, k, queries):
    ans = []
    for l, r in queries:
        seg = nums[l:r + 1]
        if len({x % k for x in seg}) > 1:
            ans.append(-1); continue
        ans.append(min(sum(abs(x - t) // k for x in seg) for t in seg))
    return ans

print([brute_all_targets(a, k, qs) for a, k, qs, _ in examples])

[[1, 2], [-1, 0, 1]]


**Nút thắt:** `O(L log L)` mỗi truy vấn ⇒ tới `4·10^4 · 4·10^4 · log` ≈ `2·10^10` — quá chậm.

## 6. Ý tưởng thuật toán / bước đột phá

Câu hỏi "phần tử nhỏ thứ `c` + tổng `c` phần tử nhỏ nhất trên đoạn **tĩnh**" là bài kinh điển của **persistent segment tree** (hoặc wavelet tree):

- Nén các thương `q_i` thành hạng `0..m−1`.
- `root[0]` = cây rỗng; `root[i+1]` = `root[i]` + chèn `q_i` (tạo `O(log m)` nút mới).
- Truy vấn: đi từ gốc của `root[r+1]` và `root[l]` cùng lúc; ở mỗi nút, số phần tử bên trái trong đoạn = `cnt[L(u)] − cnt[L(v)]`. Nếu `c` ≤ số đó thì rẽ trái, ngược lại cộng tổng bên trái vào `S_low`, trừ `c` và rẽ phải.

Mỗi truy vấn `O(log n)`, dựng `O(n log n)` thời gian và bộ nhớ.

## 7. Các cách tiếp theo

- **Cách 2:** thuật toán **Mo** (offline): sắp truy vấn theo khối, duy trì đa tập bằng hai BIT (đếm, tổng) — `O((n + q)·√n·log n)`.
- **Cách 3:** **persistent segment tree** — `O((n + q) log n)` (lời giải nộp).

### 7.1. Cách 2 — Thuật toán Mo + BIT

In [6]:
def mo_solve(nums, k, queries):
    n = len(nums)
    rem_pre = [0] * (n + 1)
    for i in range(1, n):
        rem_pre[i + 1] = rem_pre[i] + (nums[i] % k != nums[i - 1] % k)
    qv = [x // k for x in nums]
    vals = sorted(set(qv)); rank = {v: i + 1 for i, v in enumerate(vals)}; m = len(vals)
    rk = [rank[v] for v in qv]
    cnt = [0] * (m + 1); sm = [0] * (m + 1)
    LOG = 1 << (m.bit_length())

    def add(i, dc, ds):
        while i <= m:
            cnt[i] += dc; sm[i] += ds; i += i & -i

    def kth(c):                              # hạng của phần tử nhỏ thứ c và tổng các phần tử có hạng < nó
        pos = 0; acc_c = 0; acc_s = 0; step = LOG
        while step:
            nxt = pos + step
            if nxt <= m and acc_c + cnt[nxt] < c:
                pos = nxt; acc_c += cnt[nxt]; acc_s += sm[nxt]
            step >>= 1
        return pos + 1, acc_c, acc_s

    B = max(1, int(n / math.sqrt(max(1, len(queries)))))
    order = sorted(range(len(queries)), key=lambda i: (queries[i][0] // B, queries[i][1] if (queries[i][0] // B) % 2 == 0 else -queries[i][1]))
    ans = [0] * len(queries); cl, cr, total = 0, -1, 0
    for qi in order:
        l, r = queries[qi]
        while cr < r: cr += 1; add(rk[cr], 1, qv[cr]); total += qv[cr]
        while cl > l: cl -= 1; add(rk[cl], 1, qv[cl]); total += qv[cl]
        while cr > r: add(rk[cr], -1, -qv[cr]); total -= qv[cr]; cr -= 1
        while cl < l: add(rk[cl], -1, -qv[cl]); total -= qv[cl]; cl += 1
        if rem_pre[r + 1] - rem_pre[l + 1] > 0:
            ans[qi] = -1; continue
        L = r - l + 1; c = (L + 1) // 2
        pos, below_c, below_s = kth(c)
        med = vals[pos - 1]
        s_low = below_s + (c - below_c) * med
        ans[qi] = (med * c - s_low) + ((total - s_low) - med * (L - c))
    return ans

for a, k, qs, exp in examples:
    print(mo_solve(a, k, qs), "kỳ vọng", exp)

[1, 2] kỳ vọng [1, 2]
[-1, 0, 1] kỳ vọng [-1, 0, 1]


**Ghi chú:** `kth` dùng kỹ thuật "nhảy nhị phân trên BIT" (binary lifting) để tìm phần tử thứ `c` trong `O(log m)`. Mo phù hợp khi không thể dựng cấu trúc bền vững, nhưng trong Python `n√n log n` khá chậm.

### 7.2. Cách 3 — Persistent Segment Tree (lời giải chính)

In [7]:
def persistent_solve(nums, k, queries):
    n = len(nums)
    rem_pre = [0] * (n + 1)
    for i in range(1, n):
        rem_pre[i + 1] = rem_pre[i] + (nums[i] % k != nums[i - 1] % k)
    qv = [x // k for x in nums]
    vals = sorted(set(qv)); rank = {v: i for i, v in enumerate(vals)}; m = len(vals)
    Lc, Rc, C, S = [0], [0], [0], [0]               # nút 0 = cây rỗng (trỏ về chính nó)

    def insert(prev, pos, val):
        Lc.append(Lc[prev]); Rc.append(Rc[prev]); C.append(C[prev] + 1); S.append(S[prev] + val)
        root = node = len(C) - 1; p = prev; lo, hi = 0, m - 1
        while lo < hi:
            mid = (lo + hi) // 2
            if pos <= mid:
                p = Lc[p]; hi = mid
            else:
                p = Rc[p]; lo = mid + 1
            Lc.append(Lc[p]); Rc.append(Rc[p]); C.append(C[p] + 1); S.append(S[p] + val)
            child = len(C) - 1
            if pos <= mid: Lc[node] = child
            else: Rc[node] = child
            node = child
        return root

    roots = [0]
    for x in qv:
        roots.append(insert(roots[-1], rank[x], x))

    def kth_and_sum(u, v, c):
        lo, hi, acc = 0, m - 1, 0
        while lo < hi:
            mid = (lo + hi) // 2
            lc = C[Lc[u]] - C[Lc[v]]
            if c <= lc:
                u, v, hi = Lc[u], Lc[v], mid
            else:
                c -= lc; acc += S[Lc[u]] - S[Lc[v]]
                u, v, lo = Rc[u], Rc[v], mid + 1
        return vals[lo], acc + c * vals[lo]

    ans = []
    for l, r in queries:
        if rem_pre[r + 1] - rem_pre[l + 1] > 0:
            ans.append(-1); continue
        L = r - l + 1; c = (L + 1) // 2
        med, s_low = kth_and_sum(roots[r + 1], roots[l], c)
        total = S[roots[r + 1]] - S[roots[l]]
        ans.append((med * c - s_low) + ((total - s_low) - med * (L - c)))
    return ans

for a, k, qs, exp in examples:
    print(persistent_solve(a, k, qs), "kỳ vọng", exp)

[1, 2] kỳ vọng [1, 2]
[-1, 0, 1] kỳ vọng [-1, 0, 1]


## 8. TRACE từng bước

In [8]:
def trace_query(nums, k, l, r):
    seg = nums[l:r + 1]
    print(f"đoạn [{l},{r}] = {seg}, số dư mod {k}: {[x % k for x in seg]}")
    if len({x % k for x in seg}) > 1:
        print("  -> số dư khác nhau: -1"); return -1
    q = [x // k for x in seg]; qs = sorted(q)
    L = len(q); c = (L + 1) // 2; med = qs[c - 1]; s_low = sum(qs[:c]); total = sum(qs)
    print(f"  thương q = {q}, sắp xếp = {qs}, c = {c}, trung vị = {med}")
    left = med * c - s_low; right = (total - s_low) - med * (L - c)
    print(f"  S_low = {s_low}, S = {total}: chi phí trái = {left}, chi phí phải = {right}, tổng = {left + right}")
    return left + right

nums_t, k_t = [5, 11, 2, 20, 8, 14, 3], 3
for l, r in [(0, 1), (0, 5), (1, 5), (2, 6)]:
    trace_query(nums_t, k_t, l, r)
print("persistent:", persistent_solve(nums_t, k_t, [[0, 1], [0, 5], [1, 5], [2, 6]]))

đoạn [0,1] = [5, 11], số dư mod 3: [2, 2]
  thương q = [1, 3], sắp xếp = [1, 3], c = 1, trung vị = 1
  S_low = 1, S = 4: chi phí trái = 0, chi phí phải = 2, tổng = 2
đoạn [0,5] = [5, 11, 2, 20, 8, 14], số dư mod 3: [2, 2, 2, 2, 2, 2]
  thương q = [1, 3, 0, 6, 2, 4], sắp xếp = [0, 1, 2, 3, 4, 6], c = 3, trung vị = 2
  S_low = 3, S = 16: chi phí trái = 3, chi phí phải = 7, tổng = 10
đoạn [1,5] = [11, 2, 20, 8, 14], số dư mod 3: [2, 2, 2, 2, 2]
  thương q = [3, 0, 6, 2, 4], sắp xếp = [0, 2, 3, 4, 6], c = 3, trung vị = 3
  S_low = 5, S = 15: chi phí trái = 4, chi phí phải = 4, tổng = 8
đoạn [2,6] = [2, 20, 8, 14, 3], số dư mod 3: [2, 2, 2, 2, 0]
  -> số dư khác nhau: -1
persistent: [2, 10, 8, -1]


In [9]:
def trace_persistent_descent(nums, k, l, r):
    qv = [x // k for x in nums]; vals = sorted(set(qv))
    print("giá trị nén:", vals)
    c = (r - l + 2) // 2
    seg = sorted(qv[l:r + 1]); lo, hi = 0, len(vals) - 1
    while lo < hi:
        mid = (lo + hi) // 2
        lc = sum(1 for x in seg if vals[lo] <= x <= vals[mid])
        side = "trái" if c <= lc else "phải"
        print(f"  khoảng hạng [{lo},{hi}], mid={mid}, số phần tử bên trái = {lc}, c = {c} -> rẽ {side}")
        if c <= lc: hi = mid
        else: c -= lc; lo = mid + 1
    print("  trung vị =", vals[lo])

trace_persistent_descent(nums_t, k_t, 0, 5)

giá trị nén: [0, 1, 2, 3, 4, 6]
  khoảng hạng [0,5], mid=2, số phần tử bên trái = 3, c = 3 -> rẽ trái
  khoảng hạng [0,2], mid=1, số phần tử bên trái = 2, c = 3 -> rẽ phải
  trung vị = 2


## 9. Code nộp LeetCode

In [10]:
class Solution:
    def minOperations(self, nums: List[int], k: int, queries: List[List[int]]) -> List[int]:
        n = len(nums)
        bad = [0] * (n + 1)
        for i in range(1, n):
            bad[i + 1] = bad[i] + (nums[i] % k != nums[i - 1] % k)
        qv = [x // k for x in nums]
        vals = sorted(set(qv))
        rank = {v: i for i, v in enumerate(vals)}
        m = len(vals)
        Lc, Rc, C, S = [0], [0], [0], [0]

        def insert(prev, pos, val):
            Lc.append(Lc[prev]); Rc.append(Rc[prev]); C.append(C[prev] + 1); S.append(S[prev] + val)
            root = node = len(C) - 1
            p, lo, hi = prev, 0, m - 1
            while lo < hi:
                mid = (lo + hi) >> 1
                go_left = pos <= mid
                if go_left:
                    p = Lc[p]; hi = mid
                else:
                    p = Rc[p]; lo = mid + 1
                Lc.append(Lc[p]); Rc.append(Rc[p]); C.append(C[p] + 1); S.append(S[p] + val)
                child = len(C) - 1
                if go_left:
                    Lc[node] = child
                else:
                    Rc[node] = child
                node = child
            return root

        roots = [0]
        for x in qv:
            roots.append(insert(roots[-1], rank[x], x))

        ans = []
        for l, r in queries:
            if bad[r + 1] - bad[l + 1]:
                ans.append(-1)
                continue
            L = r - l + 1
            c = (L + 1) >> 1
            u, v = roots[r + 1], roots[l]
            lo, hi, acc, need = 0, m - 1, 0, c
            while lo < hi:
                mid = (lo + hi) >> 1
                lc = C[Lc[u]] - C[Lc[v]]
                if need <= lc:
                    u, v, hi = Lc[u], Lc[v], mid
                else:
                    need -= lc; acc += S[Lc[u]] - S[Lc[v]]
                    u, v, lo = Rc[u], Rc[v], mid + 1
            med = vals[lo]
            s_low = acc + need * med
            total = S[roots[r + 1]] - S[roots[l]]
            ans.append(med * c - s_low + (total - s_low) - med * (L - c))
        return ans

sol = Solution()
for a, k, qs, exp in examples:
    got = sol.minOperations(a, k, qs)
    print(got, "✔" if got == exp else "✘")

[1, 2] ✔
[-1, 0, 1] ✔


## 10. Vì sao thuật toán đúng

1. **Khả thi ⇔ cùng số dư.** (⇒) thao tác giữ số dư, nên giá trị chung cuối cùng có cùng số dư với mọi phần tử. (⇐) nếu cùng số dư `ρ`, đưa mọi phần tử về `t·k + ρ` bằng `|q_i − t|` thao tác. Kiểm tra bằng đếm số `i ∈ [l+1, r]` có số dư khác phần tử trước — bằng 0 ⇔ mọi số dư bằng nhau (bắc cầu).
2. **Tối ưu = trung vị.** Mục 3.2: `f(t) = Σ|q_i − t|` lồi, độ dốc `#{< t} − #{> t}`; tại trung vị dưới `m` (phần tử thứ `c = ⌈L/2⌉`), mọi hướng dịch chuyển đều không giảm `f`.
3. **Công thức chi phí.** `c` phần tử nhỏ nhất đều `≤ m` ⇒ tổng khoảng cách của chúng là `m·c − S_low`; `L − c` phần tử còn lại đều `≥ m` ⇒ `(S − S_low) − m·(L − c)`.
4. **Persistent segment tree.** `C[root[r+1]] − C[root[l]]` trên mỗi nút đếm đúng số phần tử của đoạn trong khoảng giá trị của nút (hiệu hai tiền tố). Đi xuống theo `c` cho phần tử thứ `c` và tổng các phần tử đứng trước nó; phần tử bằng trung vị được cộng `need·med`.

## 11. Phân tích Time / Space Complexity

| Cách | Tiền xử lý | Mỗi truy vấn | Space |
|---|---|---|---|
| 1. Sắp xếp mỗi đoạn | — | `O(L log L)` | `O(L)` |
| 2. Mo + BIT | `O(q log q)` | `O(√n log n)` trung bình | `O(n)` |
| 3. Persistent Segment Tree | `O(n log n)` | `O(log n)` | `O(n log n)` |

## 12. Bộ test

In [11]:
METHODS = [brute, brute_all_targets, mo_solve, persistent_solve, lambda a, k, q: Solution().minOperations(a, k, q)]
for a, k, qs, exp in examples:
    assert all(f(a, k, qs) == exp for f in METHODS)
print("Ví dụ đề: OK")

Ví dụ đề: OK


In [12]:
# 12.2 ca biên
assert Solution().minOperations([5], 7, [[0, 0]]) == [0]
assert Solution().minOperations([3, 3, 3], 1, [[0, 2]]) == [0]
assert Solution().minOperations([1, 10**9], 1, [[0, 1]]) == [10**9 - 1]       # k = 1: luôn khả thi
assert Solution().minOperations([2, 4, 6, 8], 2, [[0, 3], [1, 2]]) == [4, 1]
print("Ca biên: OK")

Ca biên: OK


In [13]:
# 12.3 ngẫu nhiên
rng = random.Random(3762)
for _ in range(800):
    n = rng.randint(1, 12); k = rng.randint(1, 4)
    a = [rng.randint(1, 20) for _ in range(n)]
    qs = []
    for _ in range(rng.randint(1, 10)):
        l = rng.randrange(n); qs.append([l, rng.randrange(l, n)])
    res = [f(a, k, qs) for f in METHODS]
    assert all(r == res[0] for r in res), (a, k, qs, res)
print("800 test ngẫu nhiên: OK")

800 test ngẫu nhiên: OK


In [14]:
# 12.4 test lớn: n = q = 4·10^4
rng = random.Random(1)
n = 4 * 10**4; k = 5
a = [rng.randrange(0, 10**9 // 5) * 5 + (3 if rng.random() < 0.99997 else 1) for _ in range(n)]
qs = []
for _ in range(n):
    l = rng.randrange(n); qs.append([l, min(n - 1, l + rng.randrange(0, 3000))])
t0 = time.time(); r1 = Solution().minOperations(a, k, qs); t1 = time.time()
print("thời gian persistent:", round(t1 - t0, 2), "s | số truy vấn -1:", r1.count(-1))
check = rng.sample(range(n), 300)
assert [r1[i] for i in check] == brute(a, k, [qs[i] for i in check])
print("300 truy vấn ngẫu nhiên khớp brute")

thời gian persistent: 0.72 s | số truy vấn -1: 1435
300 truy vấn ngẫu nhiên khớp brute


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Dùng **trung bình** thay cho trung vị

In [15]:
def wrong_mean(nums, k, queries):
    ans = []
    for l, r in queries:
        seg = nums[l:r + 1]
        if len({x % k for x in seg}) > 1: ans.append(-1); continue
        q = [x // k for x in seg]; t = round(sum(q) / len(q))           # SAI
        ans.append(sum(abs(x - t) for x in q))
    return ans

print("sai:", wrong_mean([1, 2, 3, 100], 1, [[0, 3]]), "| đúng:", brute([1, 2, 3, 100], 1, [[0, 3]]))

sai: [146] | đúng: [100]


### Lỗi 2 — Quên chia cho `k` (đếm hiệu giá trị thay vì số thao tác)

In [16]:
def wrong_no_div(nums, k, queries):
    ans = []
    for l, r in queries:
        seg = sorted(nums[l:r + 1])
        if len({x % k for x in seg}) > 1: ans.append(-1); continue
        m = seg[(len(seg) - 1) // 2]; ans.append(sum(abs(x - m) for x in seg))   # SAI
    return ans

print("sai:", wrong_no_div([1, 4, 7], 3, [[0, 2]]), "| đúng:", brute([1, 4, 7], 3, [[0, 2]]))

sai: [6] | đúng: [2]


### Lỗi 3 — Chỉ so số dư của hai đầu đoạn

In [17]:
def wrong_ends(nums, k, queries):
    ans = []
    for l, r in queries:
        if nums[l] % k != nums[r] % k: ans.append(-1); continue           # SAI: bỏ qua phần giữa
        q = sorted(x // k for x in nums[l:r + 1]); m = q[(len(q) - 1) // 2]
        ans.append(sum(abs(x - m) for x in q))
    return ans

print("sai:", wrong_ends([1, 2, 4], 3, [[0, 2]]), "| đúng:", brute([1, 2, 4], 3, [[0, 2]]))

sai: [1] | đúng: [-1]


### Lỗi 4 — Persistent tree: sửa nút cũ thay vì tạo nút mới

Nếu cập nhật trực tiếp `C[p] += 1` trên các nút của phiên bản trước, mọi phiên bản cũ bị thay đổi theo ⇒ hiệu `root[r+1] − root[l]` sai. Mỗi lần chèn phải **sao chép** đường đi.

In [18]:
def wrong_shared_versions(nums):
    # Mô phỏng lỗi: dùng chung MỘT mảng đếm cho mọi phiên bản (giống sửa tại chỗ)
    cnt = {}; versions = [dict(cnt)]
    shared = cnt
    snapshots = []
    for x in nums:
        shared[x] = shared.get(x, 0) + 1
        snapshots.append(shared)            # SAI: mọi phiên bản trỏ tới cùng một dict
    return [sum(s.values()) for s in snapshots]

print("số phần tử theo từng phiên bản (sai):", wrong_shared_versions([3, 1, 2]), "| đúng: [1, 2, 3]")

số phần tử theo từng phiên bản (sai): [3, 3, 3] | đúng: [1, 2, 3]


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Tăng/giảm đúng `k`" ⇒ nghĩ ngay tới **bất biến số dư** và **thương**.
- "Làm bằng nhau với chi phí `|x − t|`" ⇒ **trung vị**.
- "Phần tử thứ `c` / tổng `c` nhỏ nhất trên đoạn tĩnh, nhiều truy vấn" ⇒ persistent segment tree, wavelet tree, hoặc Mo offline.
- Kiểm tra "mọi phần tử cùng tính chất trong đoạn" ⇒ đếm **điểm thay đổi** bằng tổng tiền tố.

## 15. Biến thể của bài

1. **Không có truy vấn**, chỉ toàn mảng: sắp xếp + trung vị, `O(n log n)`.
2. **Thao tác chỉ tăng**: đích phải là max ⇒ `Σ(q_max − q_i)`; chỉ cần range max + range sum.
3. **Có cập nhật điểm**: cần cấu trúc động (BIT lồng, sqrt decomposition, hoặc offline CDQ).
4. **Chi phí bình phương** `Σ(q_i − t)²`: đích tối ưu là **trung bình** (làm tròn) — chỉ cần tổng và tổng bình phương tiền tố.

In [19]:
# Biến thể 4: chi phí bình phương — đích tối ưu là số nguyên gần trung bình nhất
def squared_cost(q):
    s = sum(q); n = len(q); best = None
    for t in (s // n, s // n + 1):
        c = sum((x - t) ** 2 for x in q); best = c if best is None else min(best, c)
    return best

rng = random.Random(15)
for _ in range(300):
    q = [rng.randint(0, 20) for _ in range(rng.randint(1, 8))]
    assert squared_cost(q) == min(sum((x - t) ** 2 for x in q) for t in range(-1, 22))
print("Biến thể 4 OK: chi phí bình phương tối ưu tại trung bình (làm tròn xuống/lên)")

Biến thể 4 OK: chi phí bình phương tối ưu tại trung bình (làm tròn xuống/lên)


## 16. So sánh các cách làm

| Cách | Ý tưởng | Mỗi truy vấn | Online? | Khi nào dùng |
|---|---|---|---|---|
| 1. Sắp xếp | Làm trực tiếp | `O(L log L)` | Có | Kiểm chứng |
| 2. Mo + BIT | Offline, dịch hai đầu | `O(√n log n)` | Không | Khi không có cấu trúc bền vững |
| 3. Persistent ST | Hiệu hai phiên bản | `O(log n)` | Có | **Lời giải chuẩn** |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output).** `nums = [2,8,5,11], k = 3`, truy vấn `[0,3]`.

**Bài 2 (tính tay).** `nums = [7,1,4,10,13], k = 3`, truy vấn `[0,4]` và `[1,3]`.

**Bài 3 (điền TODO).** Viết `kth_smallest(nums, queries)` trả về phần tử nhỏ thứ `c` của mỗi đoạn (truy vấn `[l, r, c]`) bằng persistent segment tree.

**Bài 4 (tự viết lại).** Viết hàm kiểm tra khả thi cho mọi truy vấn chỉ bằng mảng "vị trí đổi số dư gần nhất bên trái" `last[i]`.

**Bài 5 (mini challenge).** Biến thể "chỉ được **tăng**": trả lời mọi truy vấn trong `O(1)` sau tiền xử lý `O(n log n)` (sparse table cho max).

In [20]:
# Bài 3 — TODO (không gọi hàm này khi chạy notebook)
def kth_smallest(nums, queries):
    # TODO: tái sử dụng insert/kth_and_sum, bỏ phần tổng
    pass

In [21]:
# Bài 5 — TODO (không gọi hàm này khi chạy notebook)
def only_increase(nums, k, queries):
    # TODO: khả thi như cũ; chi phí = q_max * L - Σ q
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Số dư đều `2`. (2) `q = [0,2,1,3]`, trung vị dưới `1`. (3) `1 + 1 + 0 + 2 = 4`.

**Bài 2.** (1) Số dư đều `1`. (2) `q = [2,0,1,3,4]`, trung vị `2` ⇒ `0+2+1+1+2 = 6`. (3) `[1,3]`: `q = [0,1,3]` ⇒ `3`.

**Bài 3.** (1) Dựng `roots`. (2) Đi xuống theo `c`. (3) Trả `vals[lo]`.

**Bài 4.** (1) `last[i]` = chỉ số lớn nhất `j ≤ i` mà số dư `j` khác `j−1` (hoặc 0). (2) Đoạn khả thi ⇔ `last[r] ≤ l`. (3) Đối chiếu với brute.

**Bài 5.** (1) Sparse table `max` trên `q`. (2) Tổng tiền tố `q`. (3) `q_max·L − Σq`.

## 19. Đáp án tham khảo cho bài tập

In [22]:
# Bài 1, 2
assert Solution().minOperations([2, 8, 5, 11], 3, [[0, 3]]) == [4]
assert Solution().minOperations([7, 1, 4, 10, 13], 3, [[0, 4], [1, 3]]) == [6, 3] == brute([7, 1, 4, 10, 13], 3, [[0, 4], [1, 3]])
print("Bài 1, 2 OK")

Bài 1, 2 OK


In [23]:
# Bài 3, 4
def kth_smallest_sol(nums, queries):
    vals = sorted(set(nums)); rank = {v: i for i, v in enumerate(vals)}; m = len(vals)
    Lc, Rc, C = [0], [0], [0]
    def insert(prev, pos):
        Lc.append(Lc[prev]); Rc.append(Rc[prev]); C.append(C[prev] + 1)
        root = node = len(C) - 1; p, lo, hi = prev, 0, m - 1
        while lo < hi:
            mid = (lo + hi) // 2; gl = pos <= mid
            p = Lc[p] if gl else Rc[p]
            if gl: hi = mid
            else: lo = mid + 1
            Lc.append(Lc[p]); Rc.append(Rc[p]); C.append(C[p] + 1); ch = len(C) - 1
            if gl: Lc[node] = ch
            else: Rc[node] = ch
            node = ch
        return root
    roots = [0]
    for x in nums: roots.append(insert(roots[-1], rank[x]))
    out = []
    for l, r, c in queries:
        u, v, lo, hi = roots[r + 1], roots[l], 0, m - 1
        while lo < hi:
            mid = (lo + hi) // 2; lc = C[Lc[u]] - C[Lc[v]]
            if c <= lc: u, v, hi = Lc[u], Lc[v], mid
            else: c -= lc; u, v, lo = Rc[u], Rc[v], mid + 1
        out.append(vals[lo])
    return out

def feasible_last(nums, k, queries):
    last = [0] * len(nums)
    for i in range(1, len(nums)):
        last[i] = i if nums[i] % k != nums[i - 1] % k else last[i - 1]
    return [last[r] <= l for l, r in queries]

rng = random.Random(4)
for _ in range(400):
    n = rng.randint(1, 10); a = [rng.randint(1, 9) for _ in range(n)]; k = rng.randint(1, 3); qs = []; qk = []
    for _ in range(6):
        l = rng.randrange(n); r = rng.randrange(l, n); qs.append([l, r]); qk.append([l, r, rng.randint(1, r - l + 1)])
    assert kth_smallest_sol(a, qk) == [sorted(a[l:r + 1])[c - 1] for l, r, c in qk]
    assert feasible_last(a, k, qs) == [len({x % k for x in a[l:r + 1]}) == 1 for l, r in qs]
print("Bài 3, 4 OK")

Bài 3, 4 OK


In [24]:
# Bài 5: chỉ được tăng
def only_increase_sol(nums, k, queries):
    n = len(nums); q = [x // k for x in nums]
    pre = [0] * (n + 1); bad = [0] * (n + 1)
    for i in range(n): pre[i + 1] = pre[i] + q[i]
    for i in range(1, n): bad[i + 1] = bad[i] + (nums[i] % k != nums[i - 1] % k)
    sp = [q[:]]; j = 1
    while (1 << j) <= n:
        prev = sp[-1]; h = 1 << (j - 1)
        sp.append([max(prev[i], prev[i + h]) for i in range(n - (1 << j) + 1)]); j += 1
    ans = []
    for l, r in queries:
        if bad[r + 1] - bad[l + 1]: ans.append(-1); continue
        t = (r - l + 1).bit_length() - 1
        mx = max(sp[t][l], sp[t][r - (1 << t) + 1])
        ans.append(mx * (r - l + 1) - (pre[r + 1] - pre[l]))
    return ans

def only_increase_brute(nums, k, queries):
    out = []
    for l, r in queries:
        seg = nums[l:r + 1]
        if len({x % k for x in seg}) > 1: out.append(-1); continue
        m = max(seg); out.append(sum((m - x) // k for x in seg))
    return out

rng = random.Random(5)
for _ in range(400):
    n = rng.randint(1, 10); a = [rng.randint(1, 15) for _ in range(n)]; k = rng.randint(1, 3); qs = []
    for _ in range(6):
        l = rng.randrange(n); qs.append([l, rng.randrange(l, n)])
    assert only_increase_sol(a, k, qs) == only_increase_brute(a, k, qs)
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
# khả thi: bad[i+1] = bad[i] + (nums[i]%k != nums[i-1]%k); đoạn ok ⇔ bad[r+1]-bad[l+1] == 0
# q = nums // k; c = (L+1)//2; med = phần tử nhỏ thứ c; s_low = tổng c nhỏ nhất
cost = med*c - s_low + (total - s_low) - med*(L - c)
# phần tử thứ c + tổng: persistent segment tree, đi xuống root[r+1] và root[l] cùng lúc
```

**Checklist:**

- [ ] Kiểm tra số dư của **mọi** phần tử trong đoạn?
- [ ] Làm việc với thương `nums[i] // k` (đã chia `k`)?
- [ ] Dùng trung vị, không phải trung bình?
- [ ] Persistent tree tạo nút mới trên đường chèn?
- [ ] Khi tới lá, cộng `need · med` vào `s_low`?